# Data Preprocessing

原始数据每行是「一个骑手配送一组订单的 cost」。大量行共享相同订单组合，只是骑手不同。

**思路**：按订单组合聚合，压缩成 bundle → [(rider, cost), ...] 结构，预计降 10~30 倍。

In [7]:
import sys
from pathlib import Path

import pandas as pd

try:
    _here = Path(__vsc_ipynb_file__).resolve().parent
    print(f"Running in Jupyter Notebook: {_here}")
except NameError:
    _here = Path().resolve()

PROJECT_ROOT = _here.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.data import load_case_data

CASE_DIR = PROJECT_ROOT / "Case"

Running in Jupyter Notebook: C:\Users\xincy\Documents\Projects\peisong\scripts


## 1. 加载 & 概览

In [8]:
case_path = CASE_DIR / "case1.tsv"
case = load_case_data(case_path)
print(f"rows={len(case.rows):,}  orders={len(case.orders):,}  riders={len(case.riders):,}")

rows=3,813,807  orders=1,238  riders=3,675


In [9]:
df = pd.DataFrame(
    [{"orders": r.orders, "rider": r.rider, "cost": r.cost} for r in case.rows]
)
df.head(10)

,orders,rider,cost
0,"(136, 158, 572, 653)",2787,35.116164
1,"(136, 158, 572, 653)",231,48.222995
2,"(136, 158, 572, 653)",1300,48.338549
3,"(136, 158, 572, 653)",1086,63.181185
4,"(136, 158, 572, 653)",3288,72.170605
5,"(136, 158, 572, 653)",1982,74.827139
6,"(136, 158, 572, 653)",2174,75.973651
7,"(136, 158, 572, 653)",3199,80.091604
8,"(136, 158, 572, 653)",3276,81.345083
9,"(136, 158, 572, 653)",658,113.797976


## 2. 按订单组合聚合

In [10]:
bundle_df = (
    df.groupby("orders")
    .agg(
        n_riders=("rider", "nunique"),
        min_cost=("cost", "min"),
        max_cost=("cost", "max"),
        mean_cost=("cost", "mean"),
    )
    .reset_index()
)
print(f"Unique bundles: {len(bundle_df):,}  (compression: {len(df) / len(bundle_df):.1f}x)")
bundle_df.head(10)

Unique bundles: 354,681  (compression: 10.8x)


,orders,n_riders,min_cost,max_cost,mean_cost
0,"(0,)",15,1.485067,43.402011,14.779403
1,"(0, 6)",15,13.106000,182.467822,51.603986
2,"(0, 6, 9)",15,20.159389,412.406178,145.864844
3,"(0, 6, 9, 470)",15,30.998544,833.419648,281.853969
4,"(0, 6, 9, 550)",15,26.411933,661.176456,245.108661
5,"(0, 6, 9, 585)",15,27.928144,656.156067,248.022052
6,"(0, 6, 9, 745)",15,35.836900,1144.080021,423.868097
7,"(0, 6, 9, 1044)",15,38.656487,1156.984301,455.007707
8,"(0, 6, 9, 1090)",15,23.812804,975.605689,362.690073
9,"(0, 6, 9, 1155)",15,42.366722,957.207122,458.315527


In [11]:
bundle_riders = (
    df.groupby("orders")
    .apply(lambda g: list(zip(g["rider"], g["cost"])), include_groups=False)
    .reset_index()
    .rename(columns={0: "rider_cost_list"})
)
bundle_full = bundle_df.merge(bundle_riders, on="orders")
bundle_full["n_orders"] = bundle_full["orders"].apply(len)
bundle_full.head(5)

,orders,n_riders,min_cost,max_cost,mean_cost,rider_cost_list,n_orders
0,"(0,)",15,1.485067,43.402011,14.779403,"[(1819, 1.4850666666666656), (383, 3.508), (24...",1
1,"(0, 6)",15,13.106000,182.467822,51.603986,"[(383, 13.106000000000002), (2477, 14.29964444...",2
2,"(0, 6, 9)",15,20.159389,412.406178,145.864844,"[(383, 20.15938888888889), (2600, 22.014277777...",3
3,"(0, 6, 9, 470)",15,30.998544,833.419648,281.853969,"[(742, 30.99854444444445), (2600, 31.1332), (3...",4
4,"(0, 6, 9, 550)",15,26.411933,661.176456,245.108661,"[(2600, 26.411933333333337), (742, 26.58929999...",4


## 3. 分布特征

In [12]:
print("=== Orders per bundle ===")
print(bundle_full["n_orders"].value_counts().sort_index())

print("\n=== Riders per bundle ===")
print(bundle_full["n_riders"].describe())

print("\n=== Cost spread per bundle ===")
bundle_full["cost_range"] = bundle_full["max_cost"] - bundle_full["min_cost"]
print(bundle_full["cost_range"].describe())

=== Orders per bundle ===
n_orders
1      1238
2      8282
3     53593
4    291568
Name: count, dtype: int64

=== Riders per bundle ===
count    354681.000000
mean         10.752781
std           7.262006
min           1.000000
25%           3.000000
50%          15.000000
75%          15.000000
max          30.000000
Name: n_riders, dtype: float64

=== Cost spread per bundle ===
count    354681.000000
mean        271.092189
std         438.610418
min           0.000000
25%           3.334100
50%          50.629908
75%         365.843211
max        4634.557939
Name: cost_range, dtype: float64


## 4. 冲突分析

每个订单出现在多少个 bundle 里？出现越多，选择空间越大但约束越复杂。

In [13]:
from collections import Counter

order_to_bundles = Counter()
for _, row in bundle_full.iterrows():
    for o in row["orders"]:
        order_to_bundles[o] += 1

counts = Counter(order_to_bundles.values())
for n_bundles, n_orders in sorted(counts.items()):
    print(f"  {n_bundles:>3} bundles -> {n_orders:>5} orders")
print(f"\nOrders in only 1 bundle: {sum(1 for v in order_to_bundles.values() if v == 1):,}/{len(order_to_bundles):,}")

    1 bundles ->   144 orders
    2 bundles ->    16 orders
    3 bundles ->    10 orders
    4 bundles ->    11 orders
    5 bundles ->     6 orders
    6 bundles ->     6 orders
    7 bundles ->     9 orders
    8 bundles ->    14 orders
    9 bundles ->     4 orders
   10 bundles ->     5 orders
   11 bundles ->     2 orders
   12 bundles ->     5 orders
   14 bundles ->     2 orders
   15 bundles ->    13 orders
   16 bundles ->     6 orders
   17 bundles ->     4 orders
   18 bundles ->     3 orders
   19 bundles ->     7 orders
   20 bundles ->     8 orders
   21 bundles ->     1 orders
   22 bundles ->     3 orders
   23 bundles ->     5 orders
   24 bundles ->     6 orders
   25 bundles ->     2 orders
   26 bundles ->    13 orders
   27 bundles ->     2 orders
   28 bundles ->     1 orders
   29 bundles ->     4 orders
   30 bundles ->     5 orders
   31 bundles ->     4 orders
   32 bundles ->     4 orders
   33 bundles ->     2 orders
   35 bundles ->     2 orders
   36 bund

## 5. 全 Case 汇总

In [ ]:
rows = []
for p in sorted(CASE_DIR.glob("case*.tsv")):
    c = load_case_data(p)
    raw = len(c.rows)
    bundles = set(r.orders for r in c.rows)
    n_orders = len(set(o for r in c.rows for o in r.orders))
    n_riders = len(set(r.rider for r in c.rows))
    rows.append({
        "case": p.name,
        "raw_rows": raw,
        "bundles": len(bundles),
        "orders": n_orders,
        "riders": n_riders,
        "compression": f"{raw / len(bundles):.1f}x",
    })

pd.DataFrame(rows)

,case,raw_rows,bundles,orders,riders,compression
0,case1.tsv,3813807,354681,1238,3675,10.8x
1,case2.tsv,2995777,284650,1261,3518,10.5x
2,case3.tsv,2197760,231870,1237,3499,9.5x
3,case4.tsv,1777325,183706,1258,3483,9.7x
4,case5.tsv,8270852,309409,446,1734,26.7x
5,case6.tsv,8364781,278833,291,1102,30.0x
6,case7.tsv,3116345,115951,278,986,26.9x
7,case8.tsv,5877463,196471,211,1198,29.9x
8,case9.tsv,4605738,153534,209,604,30.0x


: 

## 待探索
- bundle 之间的包含关系（子集）
- 同一 bundle 内被支配的选项（cost 始终更高）
- 连通性：通过共享订单连接的 bundle 簇